# Scenario 01: Fresh-handshake latency — classical vs hybrid PQ

**Question:** At a given request rate on c7g.large, what latency overhead does `X25519MLKEM768` add over `X25519`?

**Methodology:** Each arm (`classical`, `pq-mlkem`) is run for N trials of 5 min each. Handshakes are fresh (no keep-alive, no session resumption). We report the **median-of-trial** percentile — for each trial we compute p50/p90/p99, then take the median across trials. Range across trials is shown as error bars.

**Inputs:** Gatling `simulation.log` files under `../results/YYYY-MM-DD-scenario-01-rateN/{classical,pq-mlkem}/trial-*/`.

**Outputs:**
- Per-arm percentile table (p50, p75, p90, p95, p99, p99.9)
- Overhead table: absolute (ms) and relative (%) at each percentile
- Distribution comparison chart (log-x latency, arms overlaid)
- Percentile-vs-percentile chart ("cost curve" — where does PQ hurt most?)

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import matplotlib.pyplot as plt

# --- Config ---
RESULTS_DIR = Path('../results')
# Auto-pick the most recent scenario-01 result directory
candidates = sorted([p for p in RESULTS_DIR.iterdir() if p.is_dir() and 'scenario-01' in p.name])
assert candidates, f'No scenario-01 results found under {RESULTS_DIR}'
RUN_DIR = candidates[-1]
print(f'Analyzing: {RUN_DIR.name}')

Analyzing: 2026-09-27-scenario-01-rate1000


In [5]:
def load_gatling_log(sim_log: Path) -> pd.DataFrame:
    """
    Parse a Gatling simulation.log (tab-delimited).

    Gatling 3.11 record types include:
      RUN <sim> <run-desc> <start-ms>
      USER <scenario> START <ms>
      REQUEST <scenario> <group> <request-name> <start-ms> <end-ms> <status> <message>

    We only need REQUEST lines for latency.
    """
    rows = []
    with sim_log.open() as fh:
        for line in fh:
            parts = line.rstrip('\n').split('\t')
            if not parts or parts[0] != 'REQUEST':
                continue
            # Field positions in Gatling 3.11:
            #  parts[3] = request name, parts[4] = start_ms, parts[5] = end_ms,
            #  parts[6] = status (OK/KO)
            try:
                start_ms = int(parts[4])
                end_ms   = int(parts[5])
                status   = parts[6]
                rows.append({'latency_ms': end_ms - start_ms, 'status': status})
            except (IndexError, ValueError):
                continue
    return pd.DataFrame(rows)

def load_arm(arm: str) -> list[pd.DataFrame]:
    """Load all trials for an arm, return one DataFrame per trial."""
    trials = []
    for trial_dir in sorted((RUN_DIR / arm).glob('trial-*')):
        sim_log = trial_dir / 'simulation.log'
        if not sim_log.exists():
            print(f'  ⚠️  {trial_dir.name} missing simulation.log')
            continue
        df = load_gatling_log(sim_log)
        df['arm'] = arm
        df['trial'] = int(trial_dir.name.split('-')[1])
        trials.append(df)
    return trials

classical_trials = load_arm('classical')
pq_trials        = load_arm('pq-mlkem')
print(f'Classical: {len(classical_trials)} trial(s), '
      f'{sum(len(t) for t in classical_trials):,} requests total')
print(f'PQ:        {len(pq_trials)} trial(s), '
      f'{sum(len(t) for t in pq_trials):,} requests total')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa0 in position 50: invalid start byte

In [ ]:
# --- Per-trial percentiles, then median across trials ---
PERCENTILES = [50, 75, 90, 95, 99, 99.9]

def per_trial_percentiles(trials: list[pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for t in trials:
        ok = t[t['status'] == 'OK']['latency_ms']
        row = {'trial': int(t['trial'].iloc[0]), 'n_ok': len(ok)}
        for p in PERCENTILES:
            row[f'p{p}'] = float(np.percentile(ok, p))
        rows.append(row)
    return pd.DataFrame(rows).set_index('trial')

classical_pct = per_trial_percentiles(classical_trials)
pq_pct        = per_trial_percentiles(pq_trials)
print('Classical per trial:'); print(classical_pct)
print()
print('PQ per trial:');        print(pq_pct)

In [ ]:
# --- Median across trials + range (min-max) ---
def summarise(pct_df: pd.DataFrame) -> pd.DataFrame:
    med = pct_df.median()
    lo  = pct_df.min()
    hi  = pct_df.max()
    out = pd.DataFrame({'median_ms': med, 'min_ms': lo, 'max_ms': hi})
    return out

classical_summary = summarise(classical_pct[['p50','p75','p90','p95','p99','p99.9']])
pq_summary        = summarise(pq_pct[['p50','p75','p90','p95','p99','p99.9']])

combined = pd.DataFrame({
    'classical_median_ms': classical_summary['median_ms'],
    'pq_median_ms':        pq_summary['median_ms'],
})
combined['overhead_ms']  = combined['pq_median_ms'] - combined['classical_median_ms']
combined['overhead_pct'] = 100.0 * combined['overhead_ms'] / combined['classical_median_ms']
print('Median-of-trials percentiles with PQ overhead:')
print(combined.round(2))

In [ ]:
# --- Distribution overlay: latency histogram, log scale ---
fig, ax = plt.subplots(figsize=(10, 5))

classical_all = pd.concat(classical_trials)['latency_ms']
pq_all        = pd.concat(pq_trials)['latency_ms']

bins = np.logspace(np.log10(max(1, min(classical_all.min(), pq_all.min()))),
                   np.log10(max(classical_all.max(), pq_all.max())), 60)

ax.hist(classical_all, bins=bins, alpha=0.55, label='classical (X25519)', color='#4C78A8')
ax.hist(pq_all,        bins=bins, alpha=0.55, label='pq (X25519MLKEM768)', color='#F58518')
ax.set_xscale('log')
ax.set_xlabel('Handshake+request latency (ms, log scale)')
ax.set_ylabel('Request count')
ax.set_title(f'Fresh-handshake latency distribution — {RUN_DIR.name}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_DIR / 'chart-distribution.png', dpi=150)
plt.show()
print(f'Saved: {RUN_DIR / "chart-distribution.png"}')

In [ ]:
# --- Percentile cost curve ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

xs = combined.index.astype(str)
ax1.plot(xs, combined['classical_median_ms'], 'o-', label='classical', color='#4C78A8')
ax1.plot(xs, combined['pq_median_ms'],        's-', label='pq',        color='#F58518')
ax1.set_xlabel('Percentile'); ax1.set_ylabel('Latency (ms)')
ax1.set_title('Latency by percentile')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.bar(xs, combined['overhead_pct'], color='#F58518', alpha=0.8)
ax2.set_xlabel('Percentile'); ax2.set_ylabel('PQ overhead (%)')
ax2.set_title('PQ overhead as % of classical')
ax2.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(combined['overhead_pct']):
    ax2.text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(RUN_DIR / 'chart-overhead.png', dpi=150)
plt.show()
print(f'Saved: {RUN_DIR / "chart-overhead.png"}')

In [ ]:
# --- Emit a summary JSON — publishable numbers, machine-readable ---
summary_json = {
    'run_dir':      str(RUN_DIR.name),
    'n_trials':     {'classical': len(classical_trials), 'pq_mlkem': len(pq_trials)},
    'n_requests':   {'classical': int(classical_all.count()), 'pq_mlkem': int(pq_all.count())},
    'percentiles':  combined.to_dict(orient='index'),
    'trial_ranges': {
        'classical': summarise(classical_pct[['p50','p90','p99']]).to_dict(orient='index'),
        'pq_mlkem':  summarise(pq_pct[['p50','p90','p99']]).to_dict(orient='index'),
    },
}
with open(RUN_DIR / 'summary.json', 'w') as fh:
    json.dump(summary_json, fh, indent=2, default=str)
print(f'Saved: {RUN_DIR / "summary.json"}')
print()
print('Publishable line for blog / commit:')
p50 = combined.loc[50.0]
p99 = combined.loc[99.0]
print(f'  At this rate on c7g.large: X25519MLKEM768 adds '
      f'{p50["overhead_ms"]:.2f}ms to p50 ({p50["overhead_pct"]:.1f}%) '
      f'and {p99["overhead_ms"]:.2f}ms to p99 ({p99["overhead_pct"]:.1f}%).')